In [ ]:
import cv2
import numpy as np
import open3d as o3d
import pandas as pd

# --- Calibration (based on your config) ---
image_width = 1600
image_height = 1200
fx = fy = 850.0
cx = image_width / 2
cy = image_height / 2
baseline = 0.218  # in meters

K_left = np.array([[fx, 0, cx],
                   [0, fy, cy],
                   [0,  0,  1]], dtype=np.float32)

K_right = K_left.copy()
D_left = np.zeros(5)
D_right = np.zeros(5)
R = np.eye(3)
T = np.array([[-baseline, 0, 0]], dtype=np.float32).T

# --- Load images ---
imgL_path = "./combine-2/caml/test2/picture_cam_l20_s.jpg"
imgR_path = "./combine-2/camr/test2/picture_cam_r19_s.jpg"

imgL = cv2.imread(imgL_path, cv2.IMREAD_GRAYSCALE)
imgR = cv2.imread(imgR_path, cv2.IMREAD_GRAYSCALE)
color_img = cv2.imread(imgL_path, cv2.IMREAD_COLOR)


# --- Stereo Matching ---
stereo = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=128,  # must be divisible by 16
    blockSize=9,
    P1=8*3*9**2,
    P2=32*3*9**2,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=32,
    preFilterCap=63,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)
disparity = stereo.compute(imgL, imgR).astype(np.float32) / 16.0

# --- Depth from disparity ---
depth = np.zeros_like(disparity)
depth[disparity > 0] = fx * baseline / (disparity[disparity > 0])

# --- Reproject to 3D space ---
h, w = disparity.shape
Q = np.float32([[1, 0, 0, -cx],
                [0, -1, 0, cy],
                [0, 0, 0, -fx],
                [0, 0, 1 / baseline, 0]])

points_3d = cv2.reprojectImageTo3D(disparity, Q)
mask = disparity > disparity.min()
points = points_3d[mask]
colors = color_img[mask]

# Normalize color values to [0, 1]
colors = np.clip(colors, 0, 255).astype(np.float32) / 255.0

# --- Save to Open3D Point Cloud ---
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)
o3d.io.write_point_cloud("output_stereo.ply", pcd)

# --- Save as text file ---
df = pd.DataFrame(np.hstack((points, colors * 255)), columns=["x", "y", "z", "r", "g", "b"])
df.to_csv("output_stereo.txt", index=False)

# --- Optional: Show in Open3D ---
print(f"✅ Points in cloud: {len(points)}")
o3d.visualization.draw_geometries([pcd])

In [ ]:
import open3d as o3d

# Load .ply file
#pcd = o3d.io.read_point_cloud("midas_1_cam_test2.ply")
pcd = o3d.io.read_point_cloud("output_stereo.ply")

print(pcd)
print("Number of points:", len(pcd.points))

o3d.visualization.draw_geometries([pcd])


In [ ]:
import open3d as o3d
import os

# Use the already loaded DataFrame 'df'
object_points = df[["x", "y", "z"]].values
colors = df[["r", "g", "b"]].values / 255.0

# Create Open3D PointCloud
point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(object_points)
point_cloud.colors = o3d.utility.Vector3dVector(colors)

# Visualize
o3d.visualization.draw_geometries([point_cloud], window_name='Point Cloud Visualization', width=800, height=600)

# Save
#os.makedirs('./data/output', exist_ok=True)
o3d.io.write_point_cloud('output_stereo.ply', point_cloud)
